In [86]:
import sys
sys.path.insert(0, '/home/frlab/mj_opt/mujoco/legged_robot')

import numpy as np
import mujoco
import pinocchio as pin

from core import FloatingBaseRobotState, Pinocchio_Wrapper, Mujoco_Kernel

%load_ext autoreload
%autoreload 2

np.set_printoptions(precision=4, suppress=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [87]:
xml_path = '/home/frlab/mj_opt/xmls/systems/g1/scene_29dof.xml'
urdf_path = '/home/frlab/mj_opt/xmls/systems/g1_description/g1_29dof.urdf'
package_dirs = '/home/frlab/mj_opt/xmls/robots/g1_d_description'

In [88]:
# Pinocchio_Wrapper를 먼저 생성하여 Joint 이름 목록을 추출합니다.
wrapper = Pinocchio_Wrapper(urdf_path, package_dirs)
joint_names = [wrapper.model.names[i] for i in range(2, wrapper.model.njoints)]

# 추출한 Joint 목록을 사용하여 Kernel을 생성합니다.
kernel = Mujoco_Kernel(xml_path, joint_names_pin_order=joint_names)
print(f'✅ Initialization Complete: nq={kernel.model.nq}, nv={kernel.model.nv}, nu={kernel.model.nu}')

# %% 2. 로봇 임의 자세 세팅 및 동기화
knees_bent = np.array([
    0.0, 0.0, 0.755,
    1.0, 0.0, 0.0, 0.0,  # [주의] 이 스크립트에서 넣는 qpos는 MuJoCo에 꽂히므로 wxyz 순서가 맞습니다.
    -0.312, 0.0, 0.0, 0.669, -0.363, 0.0,
    -0.312, 0.0, 0.0, 0.669, -0.363, 0.0,
    0.073, 0.0, 0.0,
])

# MuJoCo 상태 업데이트
kernel.data.qpos[:len(knees_bent)] = knees_bent
kernel.forward()

# Pinocchio로 상태 동기화 (우리가 짠 변환 로직이 여기서 돕니다)
kernel.push_to_wrapper(wrapper)

✅ Initialization Complete: nq=36, nv=35, nu=29


In [89]:
wrapper.model.gravity.linear = kernel.model.opt.gravity

In [90]:
# %% 3. Jacobian 계산 및 비교 검증

# --- Pinocchio: world-aligned Jacobian (left foot) ---
J_lin_pin, J_ang_pin = wrapper.J_world("L_foot")    # (3, 35)
J_pin_world = np.vstack([J_lin_pin, J_ang_pin])     # (6, 35)

# --- MuJoCo: Point Jacobian (left foot frame origin) ---
J_lin_mj = np.zeros((3, kernel.model.nv))
J_ang_mj = np.zeros((3, kernel.model.nv))

foot_body_id = kernel.model.body("left_ankle_roll_link").id
# [해결책 1] Body CoM이 아닌, 프레임 원점(xpos)에서의 Jacobian을 구합니다.
foot_origin_pos = kernel.data.xpos[foot_body_id] 

mujoco.mj_jac(kernel.model, kernel.data, J_lin_mj, J_ang_mj, foot_origin_pos, foot_body_id)
J_mj_world = np.vstack([J_lin_mj, J_ang_mj])        # (6, 35)

# --- [해결책 2] Base 선속도 좌표계 보정 ---
# MuJoCo(World 기준) vs Pinocchio(Body 기준)의 속도 기준을 맞춰줍니다.
# J_pin의 앞 3개 컬럼(Base Linear)에 R_body_to_world를 곱해야 MuJoCo와 수학적으로 일치합니다.
J_pin_compensated = J_pin_world.copy()
R_bw = wrapper.R_body_to_world

J_pin_compensated[:3, 0:3] = J_pin_world[:3, 0:3] @ R_bw.T
J_pin_compensated[3:, 0:3] = J_pin_world[3:, 0:3] @ R_bw.T


In [91]:
# --- 최종 비교 ---
print("=== Pinocchio vs MuJoCo Jacobian (World-Aligned, Left Foot) ===")
print(f"Pin shape: {J_pin_compensated.shape}, Mj shape: {J_mj_world.shape}")

error_matrix = J_pin_compensated - J_mj_world
print(f"Frobenius error: {np.linalg.norm(error_matrix):.6e}")
print(f"Max abs error  : {np.max(np.abs(error_matrix)):.6e}")

# 컬럼별 오차 확인
col_err = np.linalg.norm(error_matrix, axis=0)
print("\nPer-column error (first 10 DOFs):")
print(col_err[:10])

# 성공 판정
if np.max(np.abs(error_matrix)) < 1e-5:
    print("\n🎉 SUCCESS: 두 물리 엔진의 자코비안이 완벽히 일치합니다!")
else:
    print("\n⚠️ WARNING: 오차가 발생했습니다. URDF와 MJCF의 관절 위치나 축이 다를 수 있습니다.")

=== Pinocchio vs MuJoCo Jacobian (World-Aligned, Left Foot) ===
Pin shape: (6, 35), Mj shape: (6, 35)
Frobenius error: 2.423684e-08
Max abs error  : 1.388082e-08

Per-column error (first 10 DOFs):
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

🎉 SUCCESS: 두 물리 엔진의 자코비안이 완벽히 일치합니다!


In [92]:
# ==========================================
# [버그 6 검증] URDF와 MJCF의 Joint 순서 일치 확인
# ==========================================
pin_joint_names = [wrapper.model.names[i] for i in range(2, wrapper.model.njoints)]

# MuJoCo의 0번 조인트는 보통 Root(FreeFlyer)이므로 1번부터 읽습니다.
mj_joint_names  = [kernel.model.joint(i).name for i in range(1, kernel.model.njnt)]

print("Pinocchio Joint Order (first 5):", pin_joint_names[:5])
print("MuJoCo Joint Order (first 5):   ", mj_joint_names[:5])

# 만약 여기서 에러가 터진다면, M 행렬을 비교하기 전에 인덱스 Permutation 배열을 
# 만들어서 M_mj의 행과 열을 재정렬(Sorting) 해주어야 합니다.
assert pin_joint_names == mj_joint_names, \
    "치명적 오류: Pinocchio와 MuJoCo의 관절 순서가 다릅니다! 행렬 직접 비교가 불가능합니다."

Pinocchio Joint Order (first 5): ['left_hip_pitch_joint', 'left_hip_roll_joint', 'left_hip_yaw_joint', 'left_knee_joint', 'left_ankle_pitch_joint']
MuJoCo Joint Order (first 5):    ['left_hip_pitch_joint', 'left_hip_roll_joint', 'left_hip_yaw_joint', 'left_knee_joint', 'left_ankle_pitch_joint']


In [93]:
# %% 4. 동역학(Dynamics) 항 비교 검증

# --- [준비] 엄밀한 테스트를 위해 임의의 관절 속도(Velocity) 주입 ---
# 정지 상태(dq=0)에서는 코리올리(Coriolis) 효과가 0이 되므로, 
# 관절 속도에 랜덤 값을 넣어 진짜 동역학 연산이 돌아가게 만듭니다.
random_dq = np.random.randn(kernel.model.nv) * 0.5
kernel.data.qvel[:] = random_dq
kernel.forward()
kernel.push_to_wrapper(wrapper)  # 다시 동기화

print("=== 동역학(Dynamics) 비교: Pinocchio vs MuJoCo ===")

# ==========================================
# 1. 비선형 항 (Non-Linear Effects: Coriolis + Gravity)
# ==========================================
# Pinocchio: data.nle
nle_pin = wrapper.data.nle.copy()

# MuJoCo: data.qfrc_bias
nle_mj = kernel.data.qfrc_bias.copy()

# Base(6 DOF)를 제외한 Actuated Joints(29 DOF)만 비교
# (Base는 World vs Body 좌표계 차이로 인해 추가적인 회전 변환이 필요하므로 생략)
err_nle_joints = np.max(np.abs(nle_pin[6:] - nle_mj[6:]))

print("\n[1] 비선형 항 (NLE = Coriolis + Gravity)")
print(f"  Pin shape: {nle_pin.shape}, Mj shape: {nle_mj.shape}")
print(f"  Max abs error (Actuated Joints): {err_nle_joints:.6e}")


# ==========================================
# 2. 질량 행렬 / 관성 행렬 (Mass Matrix, M)
# ==========================================
# Pinocchio: 앞서 고친 compute_dynamics_term()에서 대칭화된 M을 가져옵니다.
M_pin, C_pin, g_pin, nle_pin = wrapper.compute_dynamics_term()

# MuJoCo: 1차원 배열인 qM을 2D 행렬(Full M)로 변환해주는 전용 함수 사용
M_mj = np.zeros((kernel.model.nv, kernel.model.nv))
mujoco.mj_fullM(kernel.model, M_mj, kernel.data.qM)

# Actuated Joints (29x29) 부분 추출
M_pin_joints = M_pin[6:, 6:]
M_mj_joints  = M_mj[6:, 6:]

err_M_joints = np.max(np.abs(M_pin_joints - M_mj_joints))

print("\n[2] 질량 행렬 (Mass Matrix, M)")
print(f"  Pin shape: {M_pin.shape}, Mj shape: {M_mj.shape}")
print(f"  Max abs error (Actuated Joints 29x29): {err_M_joints:.6e}")

# ==========================================
# 최종 판정
# ==========================================
if err_nle_joints < 1e-5 and err_M_joints < 1e-5:
    print("\n🎉 SUCCESS: 동역학(Dynamics) 연산이 완벽하게 일치합니다!")
    print("이제 토크 제어기(Computed Torque Control, QP 등)를 안심하고 구현하셔도 됩니다.")
else:
    print("\n⚠️ WARNING: 오차가 발생했습니다. URDF의 질량(mass)이나 관성 모멘트(inertia) 정보가 MJCF와 다를 수 있습니다.")

=== 동역학(Dynamics) 비교: Pinocchio vs MuJoCo ===

[1] 비선형 항 (NLE = Coriolis + Gravity)
  Pin shape: (35,), Mj shape: (35,)
  Max abs error (Actuated Joints): 5.792113e-01

[2] 질량 행렬 (Mass Matrix, M)
  Pin shape: (35, 35), Mj shape: (35, 35)
  Max abs error (Actuated Joints 29x29): 2.647942e-01

⚠️ WARNING: 오차가 발생했습니다. URDF의 질량(mass)이나 관성 모멘트(inertia) 정보가 MJCF와 다를 수 있습니다.


In [94]:
# 1. 재현 가능한 랜덤 속도 주입
rng = np.random.default_rng(seed=42)
random_dq = rng.standard_normal(kernel.model.nv) * 0.5
kernel.data.qvel[:] = random_dq
kernel.forward()
kernel.push_to_wrapper(wrapper)

# 2. 에러 분리 출력 (NLE 검증 부문 대체)
M_pin, C_pin, g_pin, nle_pin = wrapper.compute_dynamics_term()

# 중력 항 (qfrc_bias at dq=0)
kernel.data.qvel[:] = 0
kernel.forward()
g_mj = kernel.data.qfrc_bias.copy()

# 코리올리 항 (복구 후 계산)
kernel.data.qvel[:] = random_dq
kernel.forward()
nle_mj = kernel.data.qfrc_bias.copy()
C_mj_dq = nle_mj - g_mj  # Coriolis = NLE - Gravity

print("=== 동역학 오차 상세 분석 ===")
print(f"Gravity 오차 (Mass 불일치 확인)    : {np.max(np.abs(g_pin[6:] - g_mj[6:])):.6e}")
print(f"Coriolis 오차 (Inertia 불일치 확인) : {np.max(np.abs((nle_pin[6:] - g_pin[6:]) - C_mj_dq[6:])):.6e}")

=== 동역학 오차 상세 분석 ===
Gravity 오차 (Mass 불일치 확인)    : 7.914583e-05
Coriolis 오차 (Inertia 불일치 확인) : 9.431194e-01


In [95]:
diag_err = np.max(np.abs(np.diag(M_pin_joints) - np.diag(M_mj_joints)))
off_diag_err = np.max(np.abs((M_pin_joints - M_mj_joints) - np.diag(np.diag(M_pin_joints - M_mj_joints))))
print(f"M diag error    : {diag_err:.6e}")
print(f"M off-diag error: {off_diag_err:.6e}")


M diag error    : 1.000002e-02
M off-diag error: 2.647942e-01


In [96]:
# 양쪽에서 actuated body들의 inertia 비교
print(f"{'name':<35} {'mass_pin':>10} {'mass_mj':>10} | {'I_diag_diff':>12}")
print("-" * 90)

for i in range(1, len(wrapper.model.inertias)):
    pin_name = wrapper.model.names[i]
    pin_inertia = wrapper.model.inertias[i]
    pin_mass = pin_inertia.mass
    pin_I = pin_inertia.inertia    # 3x3, link frame 기준

    # MuJoCo 측 — body name으로 매핑
    try:
        mj_body = kernel.model.body(pin_name.replace("_joint", "_link"))
        mj_mass = mj_body.mass[0]
        mj_diag = mj_body.inertia    # 3, principal axes diagonal
        # MJCF inertia를 link frame으로 회전: I_link = R * diag(diag) * R^T
        mj_quat_iframe = mj_body.iquat   # iframe → body frame 회전 (wxyz)
        R = pin.Quaternion(mj_quat_iframe[0], mj_quat_iframe[1],
                           mj_quat_iframe[2], mj_quat_iframe[3]).toRotationMatrix()
        mj_I = R @ np.diag(mj_diag) @ R.T

        diag_diff = np.max(np.abs(np.diag(pin_I) - np.diag(mj_I)))
        full_diff = np.max(np.abs(pin_I - mj_I))
        print(f"{pin_name:<35} {pin_mass:>10.5f} {mj_mass:>10.5f} | "
              f"diag {diag_diff:>10.2e} full {full_diff:>10.2e}")
    except KeyError:
        pass


name                                  mass_pin    mass_mj |  I_diag_diff
------------------------------------------------------------------------------------------
left_hip_pitch_joint                   1.35000    1.35000 | diag   4.87e-09 full   4.87e-09
left_hip_roll_joint                    1.52000    1.52000 | diag   1.77e-09 full   1.77e-09
left_hip_yaw_joint                     1.70200    1.70200 | diag   2.89e-09 full   3.77e-09
left_knee_joint                        1.93200    1.93200 | diag   7.69e-09 full   3.35e-08
left_ankle_pitch_joint                 0.07400    0.07400 | diag   2.85e-12 full   5.47e-12
left_ankle_roll_joint                  0.60800    0.60800 | diag   1.22e-09 full   1.22e-09
right_hip_pitch_joint                  1.35000    1.35000 | diag   4.87e-09 full   4.87e-09
right_hip_roll_joint                   1.52000    1.52000 | diag   1.77e-09 full   1.77e-09
right_hip_yaw_joint                    1.70200    1.70200 | diag   2.89e-09 full   3.77e-09
right_kn

In [97]:
M_diff = M_mj - M_pin
# 어느 (i,j) 쌍이 가장 큰가
flat_idx = np.argmax(np.abs(M_diff))
i, j = np.unravel_index(flat_idx, M_diff.shape)
print(f"Max diff at ({i}, {j}) = {M_diff[i,j]:.6e}")
print(f"  pin joint name (col i): {wrapper.model.names[wrapper.model.idx_vs[wrapper.model.parents[i]] + 1] if i >= 6 else 'base_'+str(i)}")
# 위 한 줄이 복잡하면 단순화:
pin_dof_names = ['base_lin_x','base_lin_y','base_lin_z','base_ang_x','base_ang_y','base_ang_z'] + joint_names
print(f"  i={i}: {pin_dof_names[i]}")
print(f"  j={j}: {pin_dof_names[j]}")


Max diff at (1, 19) = 2.800481e+00
  pin joint name (col i): base_1
  i=1: base_lin_y
  j=19: waist_roll_joint


In [98]:
M_base_joint_diff = M_diff[:6, 6:]
print("Base-joint coupling block error:", np.max(np.abs(M_base_joint_diff)))
print("Joint-joint block error        :", np.max(np.abs(M_diff[6:, 6:])))


Base-joint coupling block error: 2.8004806223575027
Joint-joint block error        : 0.26479420340742005


In [99]:
print("=== URDF vs MJCF 조인트 축(Axis) 방향 비교 ===")
for i, pin_name in enumerate(pin_joint_names):
    # Pinocchio 조인트 축
    pin_joint_id = wrapper.model.getJointId(pin_name)
    # Pinocchio는 joint_axis가 공간 벡터로 나올 수 있으므로 z축 성분 등 확인 필요
    pin_axis = wrapper.model.joints[pin_joint_id].shortname() # 축 정보가 포함된 문자열 반환
    
    # MuJoCo 조인트 축
    mj_joint_id = mujoco.mj_name2id(kernel.model, mujoco.mjtObj.mjOBJ_JOINT, pin_name)
    mj_axis = kernel.model.jnt_axis[mj_joint_id]
    
    # 만약 축이 정확히 반대이거나(e.g., [0,1,0] vs [0,-1,0]) 아예 다르다면 여기서 발견됩니다.
    print(f"{pin_name:30} | Mj axis: {mj_axis}")

=== URDF vs MJCF 조인트 축(Axis) 방향 비교 ===
left_hip_pitch_joint           | Mj axis: [0. 1. 0.]
left_hip_roll_joint            | Mj axis: [1. 0. 0.]
left_hip_yaw_joint             | Mj axis: [0. 0. 1.]
left_knee_joint                | Mj axis: [0. 1. 0.]
left_ankle_pitch_joint         | Mj axis: [0. 1. 0.]
left_ankle_roll_joint          | Mj axis: [1. 0. 0.]
right_hip_pitch_joint          | Mj axis: [0. 1. 0.]
right_hip_roll_joint           | Mj axis: [1. 0. 0.]
right_hip_yaw_joint            | Mj axis: [0. 0. 1.]
right_knee_joint               | Mj axis: [0. 1. 0.]
right_ankle_pitch_joint        | Mj axis: [0. 1. 0.]
right_ankle_roll_joint         | Mj axis: [1. 0. 0.]
waist_yaw_joint                | Mj axis: [0. 0. 1.]
waist_roll_joint               | Mj axis: [1. 0. 0.]
waist_pitch_joint              | Mj axis: [0. 1. 0.]
left_shoulder_pitch_joint      | Mj axis: [0. 1. 0.]
left_shoulder_roll_joint       | Mj axis: [1. 0. 0.]
left_shoulder_yaw_joint        | Mj axis: [0. 0. 1.]
left_el

In [100]:
import numpy as np

print("=== 1. 로컬 스펙 검증 (조인트 원점 & 무게중심 위치) ===")
print(f"{'Joint Name':<28} | {'Pos Diff (뼈대)':<15} | {'CoM Diff (무게중심)':<15}")
print("-" * 65)

for i in range(2, wrapper.model.njoints):
    pin_name = wrapper.model.names[i]

    # Pinocchio 로컬 좌표
    pin_pos = wrapper.model.jointPlacements[i].translation
    pin_com = wrapper.model.inertias[i].lever

    try:
        # MuJoCo 매핑
        mj_joint_id = mujoco.mj_name2id(kernel.model, mujoco.mjtObj.mjOBJ_JOINT, pin_name)
        mj_body_id = kernel.model.jnt_bodyid[mj_joint_id]

        mj_pos = kernel.model.body_pos[mj_body_id]
        mj_com = kernel.model.body_ipos[mj_body_id]

        # 유클리디안 거리 오차 계산
        pos_diff = np.linalg.norm(pin_pos - mj_pos)
        com_diff = np.linalg.norm(pin_com - mj_com)

        marker = "  <-- 🚨 설계 다름!" if (pos_diff > 1e-4 or com_diff > 1e-4) else ""
        print(f"{pin_name:<28} | {pos_diff:<15.6f} | {com_diff:<15.6f}{marker}")

    except Exception as e:
        pass


print("\n=== 2. 글로벌 움직임 검증 (축 부호 반전 확인) ===")
# 임의의 자세를 취했을 때, 엔드이펙터(손/발)가 동일한 공간 좌표로 이동하는지 확인합니다.
# 만약 축 부호(+/-)가 반대라면, 글로벌 좌표(World Pos)가 완전히 다르게 찍힙니다.

# 테스트용 자세 주입
kernel.data.qpos[:len(knees_bent)] = knees_bent
kernel.forward()
kernel.push_to_wrapper(wrapper)

check_frames = ["L_foot", "R_foot", "L_hand", "R_hand"]
print(f"{'Frame':<10} | {'World Pos Diff':<15} | {'비고'}")
print("-" * 65)

for frame in check_frames:
    # Pinocchio 글로벌 위치
    pin_world_pos = wrapper.data.oMf[wrapper.fid[frame]].translation
    
    # MuJoCo 글로벌 위치 (프레임 원점 xpos)
    mj_body_name = wrapper.model.frames[wrapper.fid[frame]].name
    mj_body_id = kernel.model.body(mj_body_name).id
    mj_world_pos = kernel.data.xpos[mj_body_id]
    
    err = np.linalg.norm(pin_world_pos - mj_world_pos)
    marker = "  <-- 🚨 축 부호 반전 또는 누적 오차!" if err > 1e-3 else "  ✅ 완벽 일치"
    print(f"{frame:<10} | {err:<15.6f} | {marker}")

=== 1. 로컬 스펙 검증 (조인트 원점 & 무게중심 위치) ===
Joint Name                   | Pos Diff (뼈대)   | CoM Diff (무게중심)
-----------------------------------------------------------------
left_hip_pitch_joint         | 0.000000        | 0.000000       
left_hip_roll_joint          | 0.000000        | 0.000000       
left_hip_yaw_joint           | 0.000000        | 0.000000       
left_knee_joint              | 0.000000        | 0.000000       
left_ankle_pitch_joint       | 0.000000        | 0.000000       
left_ankle_roll_joint        | 0.000000        | 0.000000       
right_hip_pitch_joint        | 0.000000        | 0.000000       
right_hip_roll_joint         | 0.000000        | 0.000000       
right_hip_yaw_joint          | 0.000000        | 0.000000       
right_knee_joint             | 0.000000        | 0.000000       
right_ankle_pitch_joint      | 0.000000        | 0.000000       
right_ankle_roll_joint       | 0.000000        | 0.000000       
waist_yaw_joint              | 0.000000        | 0

KeyError: "Invalid name 'left_rubber_hand'. Valid names: ['left_ankle_pitch_link', 'left_ankle_roll_link', 'left_elbow_link', 'left_hip_pitch_link', 'left_hip_roll_link', 'left_hip_yaw_link', 'left_knee_link', 'left_shoulder_pitch_link', 'left_shoulder_roll_link', 'left_shoulder_yaw_link', 'left_wrist_pitch_link', 'left_wrist_roll_link', 'left_wrist_yaw_link', 'pelvis', 'right_ankle_pitch_link', 'right_ankle_roll_link', 'right_elbow_link', 'right_hip_pitch_link', 'right_hip_roll_link', 'right_hip_yaw_link', 'right_knee_link', 'right_shoulder_pitch_link', 'right_shoulder_roll_link', 'right_shoulder_yaw_link', 'right_wrist_pitch_link', 'right_wrist_roll_link', 'right_wrist_yaw_link', 'torso_link', 'waist_roll_link', 'waist_yaw_link', 'world']"

In [101]:
# Pinocchio 측 q 직접 출력
q_pin = wrapper.current_state.get_floating_base_q()
print("=== q_pin (36) ===")
print(f"base_pos:    {q_pin[0:3]}")
print(f"base_quat:   {q_pin[3:7]}  (xyzw)")
print(f"left_leg:    {q_pin[7:13]}")
print(f"right_leg:   {q_pin[13:19]}")
print(f"waist:       {q_pin[19:22]}")

# MuJoCo 측
q_mj = kernel.data.qpos.copy()
print("\n=== q_mj (36) ===")
print(f"base_pos:    {q_mj[0:3]}")
print(f"base_quat:   {q_mj[3:7]}  (wxyz)")
print(f"left_leg:    {q_mj[7:13]}")
print(f"right_leg:   {q_mj[13:19]}")
print(f"waist:       {q_mj[19:22]}")

=== q_pin (36) ===
base_pos:    [0.    0.    0.755]
base_quat:   [0. 0. 0. 1.]  (xyzw)
left_leg:    [-0.312  0.     0.     0.669 -0.363  0.   ]
right_leg:   [-0.312  0.     0.     0.669 -0.363  0.   ]
waist:       [0.073 0.    0.   ]

=== q_mj (36) ===
base_pos:    [0.    0.    0.755]
base_quat:   [1. 0. 0. 0.]  (wxyz)
left_leg:    [-0.312  0.     0.     0.669 -0.363  0.   ]
right_leg:   [-0.312  0.     0.     0.669 -0.363  0.   ]
waist:       [0.073 0.    0.   ]


In [102]:
# Pinocchio model에서 right_ankle_roll_link의 부모 체인 출력
fid = wrapper.fid["R_foot"]
frame = wrapper.model.frames[fid]
print(f"R_foot frame: {frame.name}")
print(f"  parent joint: {wrapper.model.names[frame.parent]}")

# 부모 체인 따라 올라가기
jid = frame.parent
while jid > 0:
    print(f"  joint[{jid}] = {wrapper.model.names[jid]}, parent = {wrapper.model.parents[jid]}")
    jid = wrapper.model.parents[jid]


R_foot frame: right_ankle_roll_link
  parent joint: right_ankle_roll_joint
  joint[13] = right_ankle_roll_joint, parent = 12
  joint[12] = right_ankle_pitch_joint, parent = 11
  joint[11] = right_knee_joint, parent = 10
  joint[10] = right_hip_yaw_joint, parent = 9
  joint[9] = right_hip_roll_joint, parent = 8
  joint[8] = right_hip_pitch_joint, parent = 1
  joint[1] = root_joint, parent = 0


/tmp/ipykernel_16475/3939047348.py:5: DeprecationWarning: Deprecated member. Use Frame.parentJoint instead.
  print(f"  parent joint: {wrapper.model.names[frame.parent]}")
/tmp/ipykernel_16475/3939047348.py:8: DeprecationWarning: Deprecated member. Use Frame.parentJoint instead.
  jid = frame.parent


In [103]:
# 모든 frame 이름 출력해서 right_ankle 관련 frame이 정확히 있는지 확인
for i, frame in enumerate(wrapper.model.frames):
    if "right_ankle" in frame.name or "right_foot" in frame.name.lower():
        print(f"frame[{i}]: {frame.name}, parent={wrapper.model.names[frame.parent]}")


frame[27]: right_ankle_pitch_joint, parent=right_ankle_pitch_joint
frame[28]: right_ankle_pitch_link, parent=right_ankle_pitch_joint
frame[29]: right_ankle_roll_joint, parent=right_ankle_roll_joint
frame[30]: right_ankle_roll_link, parent=right_ankle_roll_joint


/tmp/ipykernel_16475/2248759301.py:4: DeprecationWarning: Deprecated member. Use Frame.parentJoint instead.
  print(f"frame[{i}]: {frame.name}, parent={wrapper.model.names[frame.parent]}")


In [104]:
print("R_foot world pos (pin):", wrapper.data.oMf[wrapper.fid["R_foot"]].translation)
print("R_foot world pos (mj): ", kernel.data.xpos[kernel.model.body("right_ankle_roll_link").id])
print("Pelvis world pos (pin):", wrapper.data.oMf[wrapper.fid["base"]].translation)


R_foot world pos (pin): [-0.0014 -0.1185  0.0333]
R_foot world pos (mj):  [-0.0014 -0.1185  0.0333]
Pelvis world pos (pin): [0.    0.    0.755]


In [105]:
# 현재 q에서 right hip roll만 +0.1 만큼 회전
q_test = wrapper.current_state.get_floating_base_q().copy()
right_hip_roll_idx = 7 + 6 + 1   # base(7) + left_leg(6) + right_hip_pitch(0)+right_hip_roll(1) -> 14
q_test[14] += 0.1

# Pinocchio forward kinematics
pin.framesForwardKinematics(wrapper.model, wrapper.data, q_test)
r_foot_pin_after = wrapper.data.oMf[wrapper.fid["R_foot"]].translation.copy()

# MuJoCo도 같은 q 적용
q_mj_test = kernel.data.qpos.copy()
# pin q[14]는 mj q의 어디? base 7 + left_leg 6 + right_hip_pitch 1 ... = 14
# (인덱싱은 같음, 둘 다 동일 joint 순서)
q_mj_test[14] += 0.1
kernel.data.qpos[:] = q_mj_test
kernel.forward()
r_foot_mj_after = kernel.data.xpos[kernel.model.body("right_ankle_roll_link").id].copy()

print(f"R_foot after +0.1 right_hip_roll:")
print(f"  Pin: {r_foot_pin_after}")
print(f"  Mj:  {r_foot_mj_after}")
print(f"  Diff: {r_foot_pin_after - r_foot_mj_after}")


R_foot after +0.1 right_hip_roll:
  Pin: [-0.0025 -0.0669  0.0354]
  Mj:  [-0.0025 -0.0669  0.0354]
  Diff: [ 0. -0.  0.]


In [106]:
print(f"{'name':<35} {'pin_axis':<25} {'mj_axis':<25}")
print("-" * 90)

for name in joint_names:
    # Pinocchio joint axis
    pin_jid = wrapper.model.getJointId(name)
    pin_joint = wrapper.model.joints[pin_jid]
    # JointModelRevoluteUnaligned는 .axis, JointModelRX/RY/RZ는 정해진 축
    pin_axis_str = str(pin_joint).split('\n')[0]   # joint type 출력
    
    # MuJoCo joint axis  
    mj_jid = kernel.model.joint(name).id
    mj_axis = kernel.model.jnt_axis[mj_jid]
    
    print(f"{name:<35} {pin_axis_str:<25} {str(mj_axis):<25}")


name                                pin_axis                  mj_axis                  
------------------------------------------------------------------------------------------
left_hip_pitch_joint                JointModelRY              [0. 1. 0.]               
left_hip_roll_joint                 JointModelRX              [1. 0. 0.]               
left_hip_yaw_joint                  JointModelRZ              [0. 0. 1.]               
left_knee_joint                     JointModelRY              [0. 1. 0.]               
left_ankle_pitch_joint              JointModelRY              [0. 1. 0.]               
left_ankle_roll_joint               JointModelRX              [1. 0. 0.]               
right_hip_pitch_joint               JointModelRY              [0. 1. 0.]               
right_hip_roll_joint                JointModelRX              [1. 0. 0.]               
right_hip_yaw_joint                 JointModelRZ              [0. 0. 1.]               
right_knee_joint             

In [107]:
# 셀 추가
print("=== 동기화 상태 점검 ===")
print(f"kernel.data.qpos[:7]  : {kernel.data.qpos[:7]}")
print(f"wrapper q[:7]         : {wrapper.current_state.get_floating_base_q()[:7]}")
print(f"wrapper internal q[:7]: {wrapper.q_init[:7]}")    # __init__ 시점 값


=== 동기화 상태 점검 ===
kernel.data.qpos[:7]  : [0.    0.    0.755 1.    0.    0.    0.   ]
wrapper q[:7]         : [0.    0.    0.755 0.    0.    0.    1.   ]
wrapper internal q[:7]: [0.    0.    0.755 0.    0.    0.    1.   ]


In [108]:
base_fid = wrapper.fid["base"]
base_frame = wrapper.model.frames[base_fid]
print(f"base frame name        : {base_frame.name}")
print(f"base frame parent joint: {wrapper.model.names[base_frame.parent]}")
print(f"base frame placement   : {base_frame.placement}")
print(f"base frame type        : {base_frame.type}")


base frame name        : pelvis
base frame parent joint: root_joint
base frame placement   :   R =
1 0 0
0 1 0
0 0 1
  p = 0 0 0

base frame type        : BODY


/tmp/ipykernel_16475/2278377037.py:4: DeprecationWarning: Deprecated member. Use Frame.parentJoint instead.
  print(f"base frame parent joint: {wrapper.model.names[base_frame.parent]}")


In [109]:
# 강제로 updateFramePlacements 호출 후 다시 출력
pin.updateFramePlacements(wrapper.model, wrapper.data)
print(f"R_foot world pos (pin, after update): {wrapper.data.oMf[wrapper.fid['R_foot']].translation}")
print(f"Pelvis world pos (pin, after update): {wrapper.data.oMf[wrapper.fid['base']].translation}")


R_foot world pos (pin, after update): [-0.0025 -0.0669  0.0354]
Pelvis world pos (pin, after update): [0.    0.    0.755]


In [110]:
# 1. base frame 정보
base_fid = wrapper.fid["base"]
print(f"base frame parent joint id: {wrapper.model.frames[base_fid].parent}")
print(f"base frame name           : {wrapper.model.frames[base_fid].name}")

# 2. updateFramePlacements 강제 후 재확인
pin.updateFramePlacements(wrapper.model, wrapper.data)
print(f"Pelvis world (after UFP): {wrapper.data.oMf[base_fid].translation}")


base frame parent joint id: 1
base frame name           : pelvis
Pelvis world (after UFP): [0.    0.    0.755]


/tmp/ipykernel_16475/1836128238.py:3: DeprecationWarning: Deprecated member. Use Frame.parentJoint instead.
  print(f"base frame parent joint id: {wrapper.model.frames[base_fid].parent}")
